In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
books = pd.read_csv("clean_books.csv")

books.head()

,Book_ID,Title,Author,Rating,Ratings_Count,Year,Image_URL,Genre
0,41865,"Twilight (Twilight, #1)",Stephenie Meyer,3.57,3866839,2005,https://images.gr-assets.com/books/1361039443m...,Young Adult
1,2657,To Kill a Mockingbird,Harper Lee,4.25,3198671,1960,https://images.gr-assets.com/books/1361975680m...,Classics
2,4671,The Great Gatsby,F. Scott Fitzgerald,3.89,2683664,1925,https://images.gr-assets.com/books/1490528560m...,Classics
3,5907,The Hobbit,J.R.R. Tolkien,4.25,2071616,1937,https://images.gr-assets.com/books/1372847500m...,Fantasy
4,5107,The Catcher in the Rye,J.D. Salinger,3.79,2044241,1951,https://images.gr-assets.com/books/1398034300m...,Classics


In [4]:
books = books[
    [
        "Book_ID",
        "Title",
        "Author",
        "Rating",
        "Image_URL",
        "Genre"
    ]
]

books.head()

,Book_ID,Title,Author,Rating,Image_URL,Genre
0,41865,"Twilight (Twilight, #1)",Stephenie Meyer,3.57,https://images.gr-assets.com/books/1361039443m...,Young Adult
1,2657,To Kill a Mockingbird,Harper Lee,4.25,https://images.gr-assets.com/books/1361975680m...,Classics
2,4671,The Great Gatsby,F. Scott Fitzgerald,3.89,https://images.gr-assets.com/books/1490528560m...,Classics
3,5907,The Hobbit,J.R.R. Tolkien,4.25,https://images.gr-assets.com/books/1372847500m...,Fantasy
4,5107,The Catcher in the Rye,J.D. Salinger,3.79,https://images.gr-assets.com/books/1398034300m...,Classics


In [5]:
books.isnull().sum()

Book_ID      0
Title        0
Author       0
Rating       0
Image_URL    0
Genre        0
dtype: int64

In [6]:
books["Genre"] = books["Genre"].fillna("Unknown")
books["Author"] = books["Author"].fillna("")
books["Title"] = books["Title"].fillna("")

C:\Users\sejal\AppData\Local\Temp\ipykernel_174676\1292711277.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  books["Genre"] = books["Genre"].fillna("Unknown")
C:\Users\sejal\AppData\Local\Temp\ipykernel_174676\1292711277.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  books["Author"] = books["Author"].fillna("")
C:\Users\sejal\AppData\Local\Temp\ipykernel_174676\1292711277.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,co

In [8]:
books.columns.tolist()

['Book_ID', 'Title', 'Author', 'Rating', 'Image_URL', 'Genre']

In [9]:
books["features"] = (
    books["Title"] + " " +
    books["Author"] + " " +
    books["Genre"]
)

C:\Users\sejal\AppData\Local\Temp\ipykernel_174676\3946223250.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  books["features"] = (


In [10]:
books[["Title", "Genre", "features"]].head()

,Title,Genre,features
0,"Twilight (Twilight, #1)",Young Adult,"Twilight (Twilight, #1) Stephenie Meyer Young ..."
1,To Kill a Mockingbird,Classics,To Kill a Mockingbird Harper Lee Classics
2,The Great Gatsby,Classics,The Great Gatsby F. Scott Fitzgerald Classics
3,The Hobbit,Fantasy,The Hobbit J.R.R. Tolkien Fantasy
4,The Catcher in the Rye,Classics,The Catcher in the Rye J.D. Salinger Classics


In [11]:
tfidf = TfidfVectorizer(stop_words="english")
feature_vectors = tfidf.fit_transform(books["features"])

similarity = cosine_similarity(feature_vectors)

similarity.shape

(514, 514)

In [12]:
import pickle

pickle.dump(books, open("books.pkl", "wb"))
pickle.dump(similarity, open("similarity.pkl", "wb"))

In [13]:
def recommend(book_name):
    index = books[books["Title"] == book_name].index[0]

    distances = similarity[index]

    books_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in books_list:
        print(books.iloc[i[0]]["Title"])

In [14]:
books["Title"].sample(10)

359         Queen of the Darkness (The Black Jewels, #3)
231    The Burning Room (Harry Bosch, #19; Harry Bosc...
79                    The Silkworm (Cormoran Strike, #2)
310                 Something Rotten (Thursday Next, #4)
308               The Struggle (The Vampire Diaries, #2)
317                Strong Poison (Lord Peter Wimsey, #6)
235                           Bone: The Complete Edition
116                                Redwall (Redwall, #1)
103        Anne of the Island (Anne of Green Gables, #3)
221                                          Père Goriot
Name: Title, dtype: object

In [17]:
books["Title"].tolist()[:20]

['Twilight (Twilight, #1)',
 'To Kill a Mockingbird',
 'The Great Gatsby',
 'The Hobbit',
 'The Catcher in the Rye',
 'Pride and Prejudice',
 'Animal Farm',
 'Catching Fire (The Hunger Games, #2)',
 'The Fellowship of the Ring (The Lord of the Rings, #1)',
 'Harry Potter and the Chamber of Secrets (Harry Potter, #2)',
 'Harry Potter and the Goblet of Fire (Harry Potter, #4)',
 'Lord of the Flies',
 'Of Mice and Men',
 'The Lion, the Witch, and the Wardrobe (Chronicles of Narnia, #1)',
 'Eclipse (Twilight, #3)',
 'Eragon (The Inheritance Cycle, #1)',
 'Wuthering Heights',
 'Frankenstein',
 'Paper Towns',
 'One Hundred Years of Solitude']

In [18]:
recommend("Frankenstein")

Mary Poppins (Mary Poppins, #1)
The Titan's Curse (Percy Jackson and the Olympians, #3)
Where Are the Children?
A Stranger Is Watching
On the Street Where You Live
